# M3L2 E01 - LCEL Chain: componer con `|`

## Que vamos a ver

En E00 construimos un `ChatPromptTemplate`. Ahora lo conectamos con un modelo
y un parser usando **LCEL** (LangChain Expression Language).

## El concepto central: el operador `|`

LCEL permite escribir el flujo de datos de forma declarativa:

```text
prompt | llm | parser
```

Esto es exactamente lo que describe la lecture (Seccion 12):

> "LCEL hace visible el flujo de datos. En vez de muchas funciones imperativas,
> definimos una composicion."

## Este notebook necesita API key de OpenAI

Asegurate de tener `OPENAI_API_KEY` disponible antes de ejecutar las celdas con el modelo.


## Mapa de conceptos

| Concepto de la lecture | Pregunta guia | En este notebook |
|---|---|---|
| LLM wrapper | Como encapsulo el modelo? | `ChatOpenAI(model=..., temperature=0)` |
| OutputParser | Como normalizo la salida? | `StrOutputParser()` |
| LCEL | Como conecto los componentes? | `prompt \| llm \| parser` |
| Reemplazo de modelo | Puedo cambiar el modelo sin tocar el resto? | Si, cambiando solo la variable `llm` |

Flujo completo:

```text
Input
  |
  v
ChatPromptTemplate  <- formatea variables en mensajes
  |
  v
ChatOpenAI          <- genera la respuesta
  |
  v
StrOutputParser     <- extrae el texto de la respuesta
  |
  v
Respuesta (string)
```


## Scripting tactico vs ingenieria estructurada (Lecture M3L2 - Seccion 6)

La lecture M3L2 compara los dos enfoques:

| Aspecto | Script tactico (M3L1 / sin LangChain) | Pipeline orquestado (LangChain) |
|---|---|---|
| Velocidad inicial | Alta | Media |
| Mantenibilidad al crecer | Baja | Alta |
| Modularidad | Baja | Alta |
| Debugging | Manual (prints) | Estructurado (traces) |
| Reutilizacion | Baja | Alta |
| Cambio de modelo | Riesgoso | Aislar cambio en 1 linea |
| Cambio de vector store | Costoso | Intercambiar componente |
| Produccion | Fragil | Mas preparado |

**LCEL: el operador `|`** (Lecture M3L2 - Seccion 12)

LCEL significa LangChain Expression Language.
Permite componer componentes con el operador `|`:

```text
Sin LCEL (imperativo):              Con LCEL (declarativo):
---------------------------------   ----------------------------------
messages = prompt.format(...)       chain = prompt | llm | parser
ai_msg = llm.invoke(messages)
text = ai_msg.content               result = chain.invoke({"question": q})
result = parser.invoke(text)
```

El flujo con LCEL es visible en una sola linea.
El flujo imperativo requiere leer 4 lineas para entenderlo.

**Beneficios de LCEL** (Lecture M3L2 - Seccion 12.4):
- Legibilidad: el flujo de datos es visible
- Composicion: puedes combinar components
- Streaming: `.stream()` disponible automaticamente
- Batch: `.batch([q1, q2, q3])` disponible automaticamente
- Trazabilidad: compatible con LangSmith


## Bloque 1 - Sin LangChain: el script imperativo

Asi se llama al modelo directamente con la libreria `openai`.

Ejecuta esta celda y fijate en cuantos pasos manuales hay.


In [ ]:
import os
import getpass

# Cargar API key si no esta en el entorno
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OpenAI API key: ")
print("API key cargada.")


In [ ]:
# --- SIN LANGCHAIN: llamada imperativa paso a paso ---

from openai import OpenAI

client = OpenAI()

def answer_without_langchain(question: str) -> str:
    # Paso 1: construir el prompt manualmente
    system_msg = "Eres un asistente util. Responde de forma concisa."
    user_msg = question

    # Paso 2: construir los mensajes en el formato de la API
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]

    # Paso 3: llamar a la API
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0,
    )

    # Paso 4: extraer el texto de la respuesta
    answer = response.choices[0].message.content
    return answer


# Ejecutar
respuesta = answer_without_langchain("Cual es la capital de Francia?")
print(f"Respuesta: {respuesta}")
print()
print("--- Problemas de este enfoque ---")
print("1. El modelo 'gpt-4o-mini' esta hardcodeado en la funcion")
print("2. Si quiero cambiar el modelo, busco en todas las funciones")
print("3. El formato de los mensajes es especifico de OpenAI")
print("4. No hay una forma estandar de conectar este codigo con retrieval o parsers")


### El problema del hardcoding (Lecture - Seccion 9.3)

La lecture dice:

> "Configurar el modelo en un solo lugar. No hardcodear modelos en muchos archivos."

Con llamadas directas a `openai`, el nombre del modelo aparece en cada funcion.
Con `ChatOpenAI`, lo configuramos **una vez** y el resto del pipeline no sabe
(ni necesita saber) que modelo se usa.


## Bloque 2 - Con LangChain: ChatOpenAI y StrOutputParser

Primero creamos los componentes por separado para entender que hace cada uno.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LLM wrapper: encapsula el modelo
# Si quiero cambiar el modelo, cambio SOLO esta linea
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Parser: extrae el texto de la respuesta del modelo
parser = StrOutputParser()

print(f"LLM configurado: {llm.model_name}")
print(f"Parser: {type(parser).__name__}")


### TODO 1: crear el PromptTemplate

Crea un `ChatPromptTemplate` con:
- un mensaje `system` que configure el rol del asistente,
- un mensaje `human` con la variable `{question}`.

Solo una variable porque esta chain es simple: no tiene retrieval todavia.


In [ ]:
# TODO 1: crear el ChatPromptTemplate con variables {question}
prompt = None  # reemplazar con ChatPromptTemplate.from_messages([...])

print(f"Prompt creado: {type(prompt).__name__ if prompt else 'TODO no completado'}")
print(f"Variables: {prompt.input_variables if prompt else []}")


### TODO 2: componer la chain con LCEL

LCEL usa el operador `|` para conectar componentes.
El output de cada componente se convierte en el input del siguiente.

```text
chain = prompt | llm | parser
```

Esto es lo que la lecture llama "composicion declarativa" (Seccion 12).


In [ ]:
# TODO 2: componer la chain usando el operador |
# chain = prompt | llm | parser
chain = None  # reemplazar con la composicion

print(f"Chain creada: {type(chain).__name__ if chain else 'TODO no completado'}")


### TODO 3: invocar la chain

Las chains de LCEL se invocan con `.invoke(dict_con_variables)`.

Si la chain espera `{question}`, pasamos `{"question": "..."}`.


In [ ]:
# TODO 3: invocar la chain con una pregunta
# respuesta = chain.invoke({"question": "Cual es la capital de Francia?"})
# print(f"Respuesta: {respuesta}")
# print(f"Tipo de respuesta: {type(respuesta)}")
# print("StrOutputParser garantiza que la respuesta es un string simple")


## Bloque 3 - El modelo es reemplazable

Una de las ventajas clave de LangChain: si el modelo esta encapsulado,
el resto de la chain no cambia.

Ejecuta la misma chain con otro modelo (si tienes acceso) cambiando solo `llm`.


In [ ]:
# El mismo prompt y el mismo parser funcionan con cualquier ChatModel de LangChain
# Solo cambia el objeto llm

if prompt and parser:  # solo si los TODOs estan completos
    llm_v2 = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)  # mismo modelo, temperatura diferente
    chain_v2 = prompt | llm_v2 | parser

    respuesta_v2 = chain_v2.invoke({"question": "Cual es la capital de Francia?"})
    print(f"Con temperatura 0.5: {respuesta_v2}")
    print()
    print("El prompt y el parser son los mismos.")
    print("Solo cambiamos el componente LLM.")
    print("Eso es lo que la lecture llama 'reemplazo de componentes' (Seccion 7.2).")


## Bloque 4 - Checks automaticos


In [ ]:
def run_checks():
    assert prompt is not None, "TODO 1 no completado: prompt es None"
    assert chain is not None, "TODO 2 no completado: chain es None"

    # La chain devuelve un string (no un AIMessage)
    test_response = chain.invoke({"question": "Di solo la palabra 'test'"})
    assert isinstance(test_response, str), f"La respuesta debe ser str, es {type(test_response)}"
    assert len(test_response) > 0, "La respuesta no debe estar vacia"

    # La chain funciona con distintas preguntas
    r1 = chain.invoke({"question": "Cual es 2 + 2? Responde solo el numero"})
    assert "4" in r1, "La chain debe poder responder preguntas simples"

    print("M3L2 E01 Starter checks passed")


run_checks()


## Cierre - Que aprendimos

| Sin LangChain | Con LangChain (LCEL) |
|---|---|
| Llamada imperativa: `client.chat.completions.create(...)` | Composicion declarativa: `prompt \| llm \| parser` |
| Modelo hardcodeado en cada funcion. | Modelo configurado en un objeto. |
| Extraer texto: `response.choices[0].message.content` | Parser lo hace automaticamente. |
| Cambiar modelo = modificar muchos archivos. | Cambiar modelo = reemplazar el objeto `llm`. |
| El flujo esta implicito en el codigo. | El flujo es visible en la definicion de la chain. |

### Proximo paso: E02

En E02 agregamos un `Retriever` a la chain para construir el pipeline RAG completo.
